In [ ]:
%reload_ext autoreload
%autoreload 2

import json
import numpy as np
import jax
import jax.numpy as jnp

from fpp.models.np_model import NPModel
from fpp.utils.posterior import multi_corner

# Simulate data

We use `NPModel.simulate` to generate a mock dataset from a known truth dictionary,
then fit the same model back to it.

In [ ]:
m = NPModel()

truth_dict = json.load(open("../outputs/truths/truth_dict_base230927.json", "r"))
sim_map = m.simulate(truth_dict, modifiers=['deltapsf'])
data_in = jnp.array(sim_map, dtype=jnp.int32)

# Fit

We create a fresh model with the simulated data (so that `k_max` is set correctly),
then run SVI and NUTS.

In [ ]:
m = NPModel(data=data_in, psf_tag='delta')

In [ ]:
# SVI fit
m.fit_svi(
    data=data_in,
    guide='iaf', num_flows=4, hidden_dims=[64, 64],
    lr=1e-4, n_steps=5000,
)
samples_svi = m.get_svi_samples(num_samples=10000)

In [ ]:
# NUTS fit
mcmc = m.run_nuts(
    data=data_in,
    num_chains=4, num_warmup=500,
    num_samples=2500, step_size=0.05,
)
samples_nuts = m.expand_samples(mcmc.get_samples())

# Posterior

Corner plot comparing SVI vs NUTS, with the simulation truth overlaid.

In [ ]:
labels = [
    'S_pib', 'S_ics', 'S_iso', 'S_bub', 'S_gce',
    'Sps_dsk', 'Sps_gce',
    'f_bulge_poiss', 'f_bulge_ps', 'gamma_poiss', 'gamma_ps', 'C', 'zs',
]

multi_corner(
    {'SVI': samples_svi, 'NUTS': samples_nuts},
    labels,
    point_est={k: truth_dict[k] for k in labels},
    colors_dict={'SVI': 'b', 'NUTS': 'r'},
    labels_dict={'SVI': 'SVI', 'NUTS': 'NUTS'},
)